# Analiza kasnjenja letova

Notebook ucitava Parquet rezultate iz `results/` i prikazuje trazene vizualizacije.

In [ ]:
%pip install plotly pyarrow==17.0.0

     -------------------------------------- 27.8/27.8 MB 242.2 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

RESULTS_DIR = Path("../results")

routes = pd.read_parquet(RESULTS_DIR / "top_10_routes_avg_dep_delay")
cancelled = pd.read_parquet(RESULTS_DIR / "cancelled_pct_by_airline_year")
monthly = pd.read_parquet(RESULTS_DIR / "monthly_arr_delay_rolling3")
day_period = pd.read_parquet(RESULTS_DIR / "delay_frequency_by_day_period")
delay_category = pd.read_parquet(RESULTS_DIR / "delay_category_distribution")

In [2]:
routes_plot = routes.copy()
routes_plot["route"] = routes_plot["Origin"] + " -> " + routes_plot["Dest"]
fig_routes = px.bar(
    routes_plot.sort_values("avg_dep_delay", ascending=True),
    x="avg_dep_delay",
    y="route",
    orientation="h",
    title="Top 10 ruta po prosecnom kasnjenju polaska",
    labels={"avg_dep_delay": "Prosecno kasnjenje (min)", "route": "Ruta"},
)
fig_routes.show()

In [3]:
monthly_plot = monthly.sort_values(["year", "month"]).copy()
monthly_plot["period"] = monthly_plot["year"].astype(str) + "-" + monthly_plot["month"].astype(str).str.zfill(2)

fig_monthly = px.line(
    monthly_plot,
    x="period",
    y="rolling_3m_avg_arr_delay",
    markers=True,
    title="Rolling average kasnjenja dolaska (3 meseca)",
    labels={"period": "Period", "rolling_3m_avg_arr_delay": "Rolling 3m avg (min)"},
)
fig_monthly.update_layout(xaxis_tickangle=-45)
fig_monthly.show()

In [4]:
fig_delay_category = px.pie(
    delay_category,
    names="delay_category",
    values="flights_count",
    title="Distribucija delay_category",
)
fig_delay_category.show()

In [5]:
cancelled_table = cancelled.sort_values(["year", "cancelled_pct"], ascending=[True, False]).copy()
display(cancelled_table)

fig_day_period = px.bar(
    day_period.sort_values("delayed_pct", ascending=False),
    x="day_period",
    y="delayed_pct",
    title="Ucestalost kasnjenja po delu dana",
    labels={"day_period": "Deo dana", "delayed_pct": "Kasnjenje (%)"},
)
fig_day_period.show()

,year,Reporting_Airline,cancelled_pct
0,NaN,AA,0.0000
